# Q8 · Notebook 01: Build the tract-level table

**Question 8:** Does the food shelf network reach low-income, low-access (LILA) areas? Does it fill the retail gap, or repeat the retail pattern? How does that differ by place?

This notebook builds **one row per 2020 Minnesota census tract (1,502 tracts)** with everything Q8 needs:

| Block | Source |
|---|---|
| Population, low-income population, poverty, SNAP, no-vehicle, urban flag, tribal share | USDA ERS SNAP-authorized Retailer Access Map (SRAM) 2025, General Tract Characteristics |
| LILA flags, 2025 (driving and straight-line distance) | SRAM 2025 Driving / Straight Line files |
| LILA flags, 2019 (the brief's official definition) | USDA ERS Food Access Research Atlas 2019, moved to 2020 tracts with the team crosswalk |
| Where residents live | Census 2020 tract centers of population |
| Food shelf locations | `Organization.xlsx` census block → Census 2020 block internal points |
| Food shelf activity, 2024 | `org_FSStats.xlsx`, cleaned with the Q1 rules |
| Full-service grocery locations, mid-2024 | USDA FNS Historical SNAP Retailer Locator |

## Decisions (stated once, used everywhere)

1. **Food shelf sites** = sites whose 2024 reports are typed Brick and Mortar, Outreach, Tribal or College Campus. Meal programs (51 sites, ~0 households) are not grocery distribution and are excluded. Outreach includes mobile distributions; they are located at their home address, so a fixed-sites-only network is run as a check in notebook 02.
2. **Active** = reported households or pounds above zero in calendar 2024. 2024 matches Q1's window and is closest to the 2025 store data.
3. **Main LILA definition** = `LILATracts_1And10` (low income; 1 mile urban / 10 miles rural). 2025 driving-distance version is primary because it is current; the 2019 flag is reported alongside because the brief names it.
4. **Full-service grocery** = SNAP store types Supermarket, Super Store, Large Grocery Store, authorized on 1 July 2024. This matches the store types behind the USDA LILA definition.
5. **Distance** = straight-line miles from each tract's population center. Drive times were not available; straight-line distance understates rural travel, which is stated as a limitation.

In [1]:
import os, warnings, numpy as np, pandas as pd, geopandas as gpd
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

DATA = 'Data for Q8'   # the Drive folder VG → Data for Q8 (in Colab: full path under /content/drive/...)
OUT  = 'Q8_outputs'   # where outputs are written
os.makedirs(OUT, exist_ok=True)

YEAR = 2024
STORE_SNAPSHOT = pd.Timestamp('2024-07-01')
SITE_TYPES = ['Brick and Mortar Site', 'Outreach Program', 'Tribal Program', 'College Campus Program']
FIXED_TYPES = ['Brick and Mortar Site', 'Tribal Program', 'College Campus Program']
GROCERY_TYPES = ['Supermarket', 'Super Store', 'Large Grocery Store']
METRO_CORE = ['Hennepin', 'Ramsey']
METRO_SUBURBS = ['Anoka', 'Carver', 'Dakota', 'Scott', 'Washington']

def geoid11(s):
    return s.astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(11)

def haversine_miles(lat1, lon1, lat2, lon2):
    """Pairwise great-circle distance in miles: (n,) × (m,) → (n, m)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2[None, :] - lat1[:, None]
    dlon = lon2[None, :] - lon1[:, None]
    a = np.sin(dlat/2)**2 + np.cos(lat1)[:, None]*np.cos(lat2)[None, :]*np.sin(dlon/2)**2
    return 3958.8 * 2*np.arcsin(np.sqrt(a))

## 1. Tract characteristics and 2025 LILA flags (SRAM)

In [2]:
gen = pd.read_csv(f'{DATA}/2025-sram-fara-data/SRAM General Tract Characteristics Data.csv', encoding='latin-1', low_memory=False)
gen['GEOID'] = geoid11(gen.CensusTract20)
gen = gen[gen.GEOID.str[:2] == '27'].copy()

keep_gen = ['GEOID', 'County20', 'Urban', 'POP2020', 'OHU2020', 'LowIncomeTracts', 'PovertyRate',
            'MedianFamilyIncome', 'TractLOWI', 'TractKids', 'TractSeniors', 'TractHUNV', 'TractSNAP', 'TractTribalArea']
t = gen[keep_gen].copy()
t['county'] = t.County20.str.replace(' County', '', regex=False)

def sram_flags(path, prefix):
    d = pd.read_csv(path, encoding='latin-1', low_memory=False)
    d['GEOID'] = geoid11(d.CensusTract20)
    d = d[d.GEOID.str[:2] == '27']
    cols = [f'{prefix}_SRAM_LILATracts_1And10', f'{prefix}_SRAM_LILATracts_halfAnd10',
            f'{prefix}_SRAM_LILATracts_1And20', f'{prefix}_SRAM_LILATracts_Vehicle',
            f'{prefix}_SRAM_LALOWI1_10']
    cols = [c for c in cols if c in d.columns]
    return d[['GEOID'] + cols]

t = t.merge(sram_flags(f'{DATA}/2025-sram-fara-data/SRAM Driving Distance Data.csv', 'DD'), on='GEOID', how='left')
t = t.merge(sram_flags(f'{DATA}/2025-sram-fara-data/SRAM Straight Line Distance Data.csv', 'SD'), on='GEOID', how='left')
t = t.rename(columns=lambda c: c.replace('_SRAM_LILATracts_', '_lila25_').replace('_SRAM_', '_'))
print(len(t), 'tracts')
t.filter(like='lila25').sum()

1502 tracts


DD_lila25_1And10       141.0
DD_lila25_halfAnd10    291.0
DD_lila25_1And20       106.0
DD_lila25_Vehicle      104.0
SD_lila25_1And10        49.0
SD_lila25_halfAnd10    211.0
SD_lila25_1And20        39.0
SD_lila25_Vehicle       56.0
dtype: float64

## 2. 2019 LILA flags moved onto 2020 tracts

A 2020 tract is flagged if more than half of its land came from flagged 2010 tracts (`share_of_tract20` summed over flagged pieces).

In [3]:
f19 = pd.read_csv(f'{DATA}/2019_Food_Access_Research_Atlas_Data/Food Access Research Atlas.csv', encoding='latin-1', low_memory=False)
f19['GEOID10'] = geoid11(f19.CensusTract)
f19 = f19[f19.GEOID10.str[:2] == '27']
flags19 = ['LILATracts_1And10', 'LILATracts_halfAnd10', 'LILATracts_1And20', 'LILATracts_Vehicle']

xw = pd.read_csv(f'{DATA}/mn_tract_crosswalk_2010_2020.csv', dtype={'GEOID_TRACT_10': str, 'GEOID_TRACT_20': str})
m = xw.merge(f19[['GEOID10'] + flags19], left_on='GEOID_TRACT_10', right_on='GEOID10', how='left')
share = {f: (m[f].fillna(0) * m.share_of_tract20).groupby(m.GEOID_TRACT_20).sum() for f in flags19}
f19_20 = pd.DataFrame({f'lila19_{f.split("_")[1]}': (share[f] > 0.5).astype(int) for f in flags19})
f19_20.index.name = 'GEOID'
t = t.merge(f19_20.reset_index(), on='GEOID', how='left')
print('2019 flags (2010 tracts):', f19[flags19].sum().to_dict())
print('2019 flags on 2020 tracts:', t.filter(like='lila19').sum().to_dict())
print('2020 tracts with no 2019 match:', t.lila19_1And10.isna().sum())

# 2019 low-income population (≤200% of poverty, 2014-18 ACS) moved to 2020 tracts by land share
m2 = xw.merge(f19[['GEOID10', 'TractLOWI']], left_on='GEOID_TRACT_10', right_on='GEOID10', how='left')
lowi19 = (m2.TractLOWI.fillna(0) * m2.share_of_tract10).groupby(m2.GEOID_TRACT_20).sum().rename('lowinc_2019')
t = t.merge(lowi19.rename_axis('GEOID').reset_index(), on='GEOID', how='left')

2019 flags (2010 tracts): {'LILATracts_1And10': 192, 'LILATracts_halfAnd10': 361, 'LILATracts_1And20': 137, 'LILATracts_Vehicle': 141}
2019 flags on 2020 tracts: {'lila19_1And10': 217, 'lila19_halfAnd10': 393, 'lila19_1And20': 157, 'lila19_Vehicle': 164}
2020 tracts with no 2019 match: 0


### Data-quality check: the 2025 file's low-income head-count is unusable

In the 2025 SRAM file, `TractLOWI` ("low-income population") rises as poverty falls, and in many tracts the low-income population beyond 1 mile is larger than the total population beyond 1 mile. The 2025 **flags** are consistent (every tract with poverty ≥ 20% is flagged low-income), so the flags are used; the head-count is not. Demand is measured instead as **people in poverty = 2020 population × 2020–24 ACS poverty rate**, with the 2019 low-income count as a check.

In [4]:
chk = pd.DataFrame({
    'corr(TractLOWI share, poverty rate) — 2025 file': [round((t.TractLOWI / t.POP2020).corr(t.PovertyRate), 2)],
    'corr(low-income share, poverty rate) — 2019 file': [round((f19.TractLOWI / f19.Pop2010).corr(f19.PovertyRate), 2)],
    'tracts flagged low-income with poverty < 20%': [int(((t.LowIncomeTracts == 1) & (t.PovertyRate < 20)).sum())],
    'tracts with poverty ≥ 20% NOT flagged low-income': [int(((t.LowIncomeTracts == 0) & (t.PovertyRate >= 20)).sum())],
}).T.rename(columns={0: 'value'})
t['pop_poverty'] = t.POP2020 * t.PovertyRate.fillna(0) / 100
print('People in poverty, statewide: {:,.0f} ({:.1%})'.format(t.pop_poverty.sum(), t.pop_poverty.sum() / t.POP2020.sum()))
chk

People in poverty, statewide: 542,855 (9.5%)


,value
"corr(TractLOWI share, poverty rate) — 2025 file",-0.80
"corr(low-income share, poverty rate) — 2019 file",0.87
tracts flagged low-income with poverty < 20%,404.00
tracts with poverty ≥ 20% NOT flagged low-income,0.00


## 3. Where residents live: tract population centers

In [5]:
cp = pd.read_csv(f'{DATA}/CenPop2020_Mean_TR27.txt', dtype=str, encoding='utf-8-sig')
cp['GEOID'] = cp.STATEFP + cp.COUNTYFP + cp.TRACTCE
cp['lat'] = cp.LATITUDE.astype(float); cp['lon'] = cp.LONGITUDE.astype(float)
t = t.merge(cp[['GEOID', 'lat', 'lon']], on='GEOID', how='left')
print('tracts without a population center:', t.lat.isna().sum())

tracts without a population center: 0


## 4. Food shelf activity in 2024, cleaned with the Q1 rules

Same rules as `Q1/cleaning_log.md`: DateForSearch is the date; duplicate site-months keep the larger row; households greater than individuals are rebuilt from the site's own persons-per-household ratio.

In [6]:
fs = pd.read_excel(f'{DATA}/org_FSStats.xlsx')
fs['date'] = pd.to_datetime(fs.DateForSearch)
fs = (fs.sort_values(['_id_site', 'date', 'IndividualsReg', 'PoundsReg', '__ID'], ascending=[True, True, False, False, False])
        .drop_duplicates(['_id_site', 'date'], keep='first'))

# households > individuals: rebuild from the site's clean persons-per-household ratio
ok = (fs.HouseholdsReg > 0) & (fs.IndividualsReg >= fs.HouseholdsReg)
ratio_site = (fs[ok].IndividualsReg / fs[ok].HouseholdsReg).groupby(fs[ok]._id_site).median()
ratio_state = (fs[ok].IndividualsReg / fs[ok].HouseholdsReg).median()
bad = (fs.HouseholdsReg > fs.IndividualsReg) & (fs.IndividualsReg > 0)
r = fs.loc[bad, '_id_site'].map(ratio_site).fillna(ratio_state).clip(lower=1.5)
fs.loc[bad, 'HouseholdsReg'] = (fs.loc[bad, 'IndividualsReg'] / r).round()
print('household rows rebuilt:', int(bad.sum()))

f24 = fs[fs.date.dt.year == YEAR]
site = (f24.groupby('_id_site')
          .agg(site_type=('AgencyType', lambda x: x.mode().iat[0] if x.notna().any() else None),
               households=('HouseholdsReg', 'sum'), individuals=('IndividualsReg', 'sum'),
               pounds=('PoundsReg', 'sum'), months=('date', 'nunique'))
          .reset_index())
site = site[(site.households > 0) | (site.pounds > 0)]
print('sites with activity in', YEAR, ':', len(site))
site.site_type.value_counts(dropna=False)

household rows rebuilt: 149
sites with activity in 2024 : 510


site_type
Brick and Mortar Site     393
Outreach Program           69
NaN                        38
College Campus Program      5
Tribal Program              4
Meal Program                1
Name: count, dtype: int64

## 5. Food shelf locations

`Organization.block_geoid` is a 2020 census block. Its internal point (from the Census block file's `.dbf`, no geometry needed) gives the site's latitude and longitude.

In [7]:
org = pd.read_excel(f'{DATA}/Organization.xlsx')
org['block'] = org.block_geoid.astype('Int64').astype(str).str.zfill(15).where(org.block_geoid.notna())
blk = gpd.read_file(f'{DATA}/tl_2024_27_tabblock20/tl_2024_27_tabblock20.dbf', ignore_geometry=True,
                    columns=['GEOID20', 'INTPTLAT20', 'INTPTLON20'])
blk['s_lat'] = blk.INTPTLAT20.astype(float); blk['s_lon'] = blk.INTPTLON20.astype(float)

site = site.merge(org[['__ID', 'AgencyType', 'AgencySubType', 'City', 'County', 'Region', 'SuperRegion',
                       'block', 'is_zip_centroid', 'zcountDaysOperations']],
                  left_on='_id_site', right_on='__ID', how='left')
site['site_type'] = site.site_type.fillna(site.AgencyType)
site = site.merge(blk[['GEOID20', 's_lat', 's_lon']], left_on='block', right_on='GEOID20', how='left')
site['in_network'] = site.site_type.isin(SITE_TYPES)
site['fixed_site'] = site.site_type.isin(FIXED_TYPES) & (site.AgencySubType.fillna('') != 'Mobile')
site['located'] = site.s_lat.notna()
site['site_tract'] = site.block.str[:11]

net = site[site.in_network]
print('network sites active in 2024:', len(net), '| located:', int(net.located.sum()),
      '| share of 2024 pounds located: {:.1%}'.format(net[net.located].pounds.sum() / net.pounds.sum()))
net = net[net.located].copy()
net.to_csv(f'{OUT}/q8_food_shelf_sites_2024.csv', index=False)
net.site_type.value_counts()

network sites active in 2024: 471 | located: 466 | share of 2024 pounds located: 99.9%


site_type
Brick and Mortar Site     389
Outreach Program           68
College Campus Program      5
Tribal Program              4
Name: count, dtype: int64

## 6. Full-service grocery stores on 1 July 2024

In [8]:
snap = pd.read_csv(f'{DATA}/Historical SNAP Retailer Locator Data 2005-2025.csv', dtype=str, encoding='latin-1')
snap = snap[snap.State == 'MN'].copy()
snap['auth'] = pd.to_datetime(snap['Authorization Date'], format='%m/%d/%Y', errors='coerce')
snap['end']  = pd.to_datetime(snap['End Date'], format='%m/%d/%Y', errors='coerce')
active = snap[(snap.auth <= STORE_SNAPSHOT) & (snap.end.isna() | (snap.end > STORE_SNAPSHOT))]
groc = active[active['Store Type'].isin(GROCERY_TYPES)].copy()
groc['g_lat'] = groc.Latitude.astype(float); groc['g_lon'] = groc.Longitude.astype(float)
anysnap = active[~active['Store Type'].isin(["Farmers' Market", 'Delivery Route'])].copy()
print('MN SNAP stores active on', STORE_SNAPSHOT.date(), ':', len(active), '| full-service grocery:', len(groc))
groc[['Store Name', 'Store Type', 'City', 'County', 'g_lat', 'g_lon']].to_csv(f'{OUT}/q8_grocery_stores_2024.csv', index=False)

MN SNAP stores active on 2024-07-01 : 3634 | full-service grocery: 796


## 7. Distances from each tract's population center

Miles to the nearest food shelf (all network sites, and fixed sites only), the nearest full-service grocery, and any SNAP store; counts of food shelves within 5, 10 and 20 miles.

In [9]:
tl = t.lat.values; tn = t.lon.values
D_fs = haversine_miles(tl, tn, net.s_lat.values, net.s_lon.values)
D_fx = D_fs[:, net.fixed_site.values]
D_gr = haversine_miles(tl, tn, groc.g_lat.values, groc.g_lon.values)
D_sn = haversine_miles(tl, tn, anysnap.Latitude.astype(float).values, anysnap.Longitude.astype(float).values)

t['mi_foodshelf'] = D_fs.min(1)
t['mi_foodshelf_fixed'] = D_fx.min(1)
t['mi_grocery'] = D_gr.min(1)
t['mi_any_snap_store'] = D_sn.min(1)
for r_ in (5, 10, 20):
    t[f'fs_within_{r_}mi'] = (D_fs <= r_).sum(1)
nearest = D_fs.argmin(1)
t['nearest_site_id'] = net._id_site.values[nearest]
t['nearest_site_city'] = net.City.values[nearest]
t[['mi_foodshelf', 'mi_foodshelf_fixed', 'mi_grocery', 'mi_any_snap_store']].describe().round(2)

,mi_foodshelf,mi_foodshelf_fixed,mi_grocery,mi_any_snap_store
count,1502.00,1502.00,1502.00,1502.00
mean,2.97,3.14,2.49,1.45
std,3.56,3.74,3.47,2.07
min,0.01,0.01,0.03,0.01
25%,0.84,0.89,0.58,0.33
50%,1.70,1.78,1.04,0.61
75%,3.64,3.96,2.56,1.52
max,33.82,33.82,29.60,15.31


## 8. Food shelf capacity reachable from each tract (two-step floating catchment)

A standard spatial-access measure (2SFCA). Step 1: each site's 2024 pounds (or household visits) are divided by the people in poverty living within the catchment radius. Step 2: each tract adds up those ratios over the sites within the same radius. The result reads as **pounds a year available per person in poverty within reach**. Radius: 10 miles (5 and 20 as checks). The same is computed with the 2019 low-income count as weights.

In [10]:
def tsfca(D, supply, weight, radius):
    within = D <= radius
    demand = (within * weight[:, None]).sum(0)
    R = np.divide(supply, demand, out=np.zeros_like(supply, dtype=float), where=demand > 0)
    return (within * R[None, :]).sum(1)

w_pov = t.pop_poverty.fillna(0).values
w_l19 = t.lowinc_2019.fillna(0).values
lbs = net.pounds.values.astype(float); hh = net.households.values.astype(float)
for r_ in (5, 10, 20):
    t[f'lbs_per_poor_{r_}mi'] = tsfca(D_fs, lbs, w_pov, r_)
    t[f'visits_per_poor_{r_}mi'] = tsfca(D_fs, hh, w_pov, r_)
t['lbs_per_lowinc19_10mi'] = tsfca(D_fs, lbs, w_l19, 10)
t[['lbs_per_poor_10mi', 'visits_per_poor_10mi', 'lbs_per_lowinc19_10mi']].describe().round(1)

,lbs_per_poor_10mi,visits_per_poor_10mi,lbs_per_lowinc19_10mi
count,1502.0,1502.0,1502.0
mean,281.3,4.9,114.3
std,158.3,3.1,63.1
min,0.0,0.0,0.0
25%,160.0,2.5,65.3
50%,289.9,4.7,117.0
75%,381.5,7.2,162.5
max,1256.3,22.8,425.2


## 9. Place groups and derived flags

In [11]:
def place(row):
    if row.county in METRO_CORE:   return 'Metro core (Hennepin, Ramsey)'
    if row.county in METRO_SUBURBS: return 'Metro suburbs'
    return 'Greater MN, urban' if row.Urban == 1 else 'Greater MN, rural'
t['place'] = t.apply(place, axis=1)
t['tribal'] = (t.TractTribalArea.fillna(0) >= 50).astype(int)   # TractTribalArea is a percentage (0–100)
t['low_income'] = t.LowIncomeTracts.fillna(0).astype(int)
t['lila'] = t.DD_lila25_1And10.fillna(0).astype(int)        # primary flag
t['lila_2019'] = t.lila19_1And10.fillna(0).astype(int)
t['la_threshold_mi'] = np.where(t.Urban == 1, 1.0, 10.0)    # USDA 1-mile urban / 10-mile rural
t['fs_within_la_threshold'] = (t.mi_foodshelf <= t.la_threshold_mi).astype(int)
t['pct_no_vehicle'] = 100 * t.TractHUNV / t.OHU2020
t['pct_snap_hh'] = 100 * t.TractSNAP / t.OHU2020
t['group'] = np.select([t.lila == 1, t.low_income == 1], ['LILA', 'Low income, not low access'], 'Not low income')

t.to_csv(f'{OUT}/q8_tract_table.csv', index=False)
print(len(t), 'tracts written to', f'{OUT}/q8_tract_table.csv')
t.groupby('group').agg(tracts=('GEOID', 'size'), people_in_poverty=('pop_poverty', 'sum'), lowinc_2019=('lowinc_2019', 'sum')).round(0)

1502 tracts written to Q8_outputs/q8_tract_table.csv


,tracts,people_in_poverty,lowinc_2019
group,,,
LILA,141,85855.0,176622.0
"Low income, not low access",397,220084.0,490815.0
Not low income,964,236917.0,661504.0
